In [1]:
import math


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


import torchvision
from torchvision import transforms

In [2]:
train_dataset = torchvision.datasets.ImageFolder(
    "datasets/butterflies",
    transform=transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ]
    ),
)

In [3]:
class PatchEmbed(nn.Module):
    """ """

    def __init__(
        self,
        img_size: int | tuple[int, int] = 224,
        patch_size: int | tuple[int, int] = 16,
        in_chans: int = 3,
        embed_dim: int = 768,
        bias: bool = True,
    ):
        super().__init__()

        if isinstance(img_size, int):
            img_size = (img_size, img_size)
        if isinstance(patch_size, int):
            patch_size = (patch_size, patch_size)

        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = tuple(
            [
                s // p
                for s, p in zip(
                    img_size,
                    patch_size,
                )
            ]
        )

        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=bias,
        )

    def forward(self, x: torch.Tensor):
        # (B, C, H, W) <= x

        # B: batch
        # C: channel
        # H: height (original)
        # W: width (original)

        # D: embed_dim
        # h: height (projected)
        # w: width (projected)
        # L: seqlen = h*w

        # (B, D, h, w)
        x = self.proj(x)

        # (B, D, L)
        # (B, L, D)
        x = x.flatten(2).transpose(1, 2)

        return x


In [4]:
patch_embed = PatchEmbed()

img_tensor, label = train_dataset[0]
image_batch = img_tensor[None, :]
print(image_batch.shape)
image_projected = patch_embed(image_batch)
print(image_projected.shape)
print(patch_embed.grid_size)
print(patch_embed.num_patches)

torch.Size([1, 3, 224, 224])
torch.Size([1, 196, 768])
(14, 14)
196


In [5]:
class TimestepEmbedder(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        frequency_embedding_size: int = 256,
        use_bias=True,
    ):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=use_bias),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=use_bias),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(
        t: torch.Tensor,
        dim: int,
        max_period: int = 10000,
        freq_scale: int = 1,
    ):
        assert dim % 2 == 0

        half = dim // 2
        freqs = freq_scale * torch.exp(
            #####
            -math.log(max_period) * torch.arange(0, half, dtype=torch.float32) / half
        ).to(
            device=t.device,
        )

        #     t: (B,)
        # freqs: (half,)
        #  args: (B, half)
        args = t[:, None].float() * freqs[None, :]
        # embedding: (B, dim)
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return embedding

    def forward(
        self,
        t: torch.Tensor,
        freq_scale: int = 1,
    ):
        #      t: (B,)
        # t_freq: (B, frequency_embedding_size)
        #  t_emb: (B, D)
        t_freq = self.timestep_embedding(
            t, self.frequency_embedding_size, freq_scale=freq_scale
        )
        t_emb = self.mlp(t_freq)
        return t_emb


In [6]:
timestep_embedder = TimestepEmbedder(768)
timestep_embedder(
    torch.tensor(
        [0, 100, 900],
    )
)

tensor([[-0.0042, -0.2243, -0.1150,  ..., -0.0304,  0.0076,  0.1643],
        [ 0.0735, -0.2667,  0.0768,  ..., -0.0979,  0.0649,  0.0874],
        [ 0.0608, -0.0525,  0.1020,  ..., -0.0352, -0.1571, -0.0558]],
       grad_fn=<AddmmBackward0>)

In [7]:
class LabelEmbedder(nn.Module):
    def __init__(
        self,
        num_classes: int,
        hidden_size: int,
        dropout_prob: float,
    ):
        super().__init__()
        use_cfg_embedding = 1 if (dropout_prob > 0.0) else 0
        self.embedding_table = nn.Embedding(
            num_classes + use_cfg_embedding,
            hidden_size,
        )
        self.num_classes = num_classes
        self.dropout_prob = dropout_prob

    def token_drop(
        self,
        labels: torch.Tensor,
        force_drop_mask: torch.Tensor = None,
    ):
        # force_drop_mask: 1D mask to labels
        if force_drop_mask is None:
            drop_mask = (
                torch.rand(labels.shape[0], device=labels.device) < self.dropout_prob
            )
        else:
            assert force_drop_mask.shape == labels.shape
            drop_mask = force_drop_mask.bool()

        labels = torch.where(drop_mask, self.num_classes, labels)
        return labels

    def forward(
        self,
        labels: torch.Tensor,
        train: bool,
        force_drop_mask: torch.Tensor = None,
    ):
        use_dropout = (
            #####
            self.dropout_prob > 0
        )
        if (train and use_dropout) or (force_drop_mask is not None):
            labels = self.token_drop(
                labels=labels,
                force_drop_mask=force_drop_mask,
            )
        embeddings = self.embedding_table(labels)
        return embeddings


In [8]:
label_embedder = LabelEmbedder(
    num_classes=20,
    hidden_size=768,
    dropout_prob=0.8,
)
label_embedder(
    torch.arange(0, 20),
    train=True,
    # force_drop_mask=torch.tensor([True] * 20),
)

tensor([[-1.4337,  0.1466,  0.8834,  ...,  1.5827, -0.1285, -0.5668],
        [-1.4337,  0.1466,  0.8834,  ...,  1.5827, -0.1285, -0.5668],
        [-1.4337,  0.1466,  0.8834,  ...,  1.5827, -0.1285, -0.5668],
        ...,
        [-1.4337,  0.1466,  0.8834,  ...,  1.5827, -0.1285, -0.5668],
        [-1.4337,  0.1466,  0.8834,  ...,  1.5827, -0.1285, -0.5668],
        [-1.4337,  0.1466,  0.8834,  ...,  1.5827, -0.1285, -0.5668]],
       grad_fn=<EmbeddingBackward0>)

In [9]:
class Attention(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        n_heads: int = 8,
        qkv_bias: bool = True,
        proj_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
    ):
        super().__init__()
        assert embed_dim % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.scale = self.head_dim ** (-0.5)

        self.w_qkv = nn.Linear(embed_dim, embed_dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(embed_dim, embed_dim, bias=proj_bias)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, embed_dim = x.shape

        # (B, L, embed_dim)
        # (B, L, 3, n_heads, head_dim)
        # (3, B, n_heads, L, head_dim)
        qkv: torch.Tensor = (
            self.w_qkv(x)
            .reshape(B, L, 3, self.n_heads, self.head_dim)
            .permute(2, 0, 3, 1, 4)
        )

        # (B, n_heads, L, head_dim)
        q, k, v = qkv.unbind(0)
        x = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=(self.attn_drop.p if self.training else 0.0),
        )
        x = x.transpose(1, 2).reshape(B, L, embed_dim)

        x = self.proj(x)
        x = self.proj_drop(x)

        return x


class Mlp(nn.Module):
    def __init__(
        self,
        in_features: int,
        hidden_features: int = None,
        out_features: int = None,
        bias: bool | tuple[bool, bool] = True,
        drop_probs: float | tuple[float, float] = 0.0,
    ):
        super().__init__()

        if hidden_features is None:
            hidden_features = in_features

        if out_features is None:
            out_features = in_features

        if isinstance(bias, bool):
            bias = (bias, bias)

        if isinstance(drop_probs, float):
            drop_probs = (drop_probs, drop_probs)

        self.fc1 = nn.Linear(in_features, hidden_features, bias=bias[0])
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(drop_probs[0])
        self.fc2 = nn.Linear(hidden_features, out_features, bias=bias[1])
        self.drop2 = nn.Dropout(drop_probs[1])

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x


In [10]:
def modulate(x, shift, scale):
    #     x: (B, L, D)
    # shift: (B, D)
    # scale: (B, D)
    return x * (1 + scale[:, None]) + shift[:, None]


class DiTBlock(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        n_heads: int,
        mlp_ratio: float = 4.0,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = Attention(hidden_size, n_heads=n_heads)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)

        mlp_hidden_dim = int(hidden_size * mlp_ratio)
        self.mlp = Mlp(
            in_features=hidden_size,
            hidden_features=mlp_hidden_dim,
            drop_probs=0.0,
        )

        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True),
        )

        # nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        # nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

    def forward(
        self,
        x: torch.Tensor,
        c: torch.Tensor,
    ):
        # x: (B, L, D)
        # c: (B, D)

        # (B, D)
        (
            shift_msa,
            scale_msa,
            gate_msa,
            #####
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.adaLN_modulation(c).chunk(6, dim=1)

        x = x + gate_msa[:, None] * self.attn(
            modulate(self.norm1(x), shift_msa, scale_msa)
        )
        x = x + gate_mlp[:, None] * self.mlp(
            modulate(self.norm2(x), shift_mlp, scale_mlp)
        )

        return x


In [11]:
class FinalLayer(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        patch_size: int,
        out_channels: int,
    ):
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                hidden_size,
                2 * hidden_size,
                bias=True,
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
        c: torch.Tensor,
    ):
        # x: (B, L, D)
        # c: (B, D)

        # (B, D)
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        x = modulate(self.norm_final(x), shift, scale)
        x = self.linear(x)
        return x


class NanoDiT(nn.Module):
    def __init__(
        self,
        input_size: int = 32,
        patch_size: int = 2,
        in_channels: int = 3,
        hidden_size: int = 384,
        depth: int = 12,
        n_heads: int = 6,
        mlp_ratio: float = 4.0,
        num_classes: int = 10,
        timestep_freq_scale: float = 1.0,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = in_channels
        self.patch_size = patch_size
        self.n_heads = n_heads
        self.timestep_freq_scale = timestep_freq_scale

        self.x_embedder = PatchEmbed(
            img_size=input_size,
            patch_size=patch_size,
            in_chans=in_channels,
            embed_dim=hidden_size,
        )

        self.t_embedder = TimestepEmbedder(
            hidden_size=hidden_size,
        )

        self.y_embedder = LabelEmbedder(
            num_classes=num_classes,
            hidden_size=hidden_size,
            dropout_prob=0.1,
        )

        num_patches = self.x_embedder.num_patches

        # fixed sin-cos pos embed
        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches, hidden_size),
            requires_grad=False,
        )

        self.blocks = nn.ModuleList(
            [
                DiTBlock(
                    hidden_size=hidden_size,
                    n_heads=n_heads,
                    mlp_ratio=mlp_ratio,
                )
                for _ in range(depth)
            ]
        )
        self.final_layer = FinalLayer(
            hidden_size=hidden_size,
            patch_size=patch_size,
            out_channels=self.out_channels,
        )

        self.initialize_weights()

    def initialize_weights(self):
        def _basic_init(module):
            if isinstance(module, nn.Linear):
                torch.nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

        self.apply(_basic_init)

        # init and freeze pos_embed
        pos_embed = get_2d_sincos_pos_embed(
            self.pos_embed.shape[-1],
            int(self.x_embedder.num_patches**0.5),
        )
        self.pos_embed.data.copy_(
            #####
            torch.from_numpy(pos_embed).float()[None, :]
        )

        # init patch_embed like nn.Linear (instead of nn.Conv2d)
        w = self.x_embedder.proj.weight.data
        nn.init.xavier_normal_(
            w.view(
                [
                    w.shape[0],
                    -1,
                ]
            )
        )

        # init label embedding table
        nn.init.normal_(self.y_embedder.embedding_table.weight, std=0.02)

        # init timestep embedding MLP
        nn.init.normal_(self.t_embedder.mlp[0].weight, std=0.02)
        nn.init.normal_(self.t_embedder.mlp[2].weight, std=0.02)

        # zero-out adaLN modulation
        for block in self.blocks:
            nn.init.constant_(block.adaLN_modulation[-1].weight, 0)
            nn.init.constant_(block.adaLN_modulation[-1].bias, 0)

        # zero-out output layer
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].bias, 0)
        nn.init.constant_(self.final_layer.linear.weight, 0)
        if getattr(self.final_layer.linear, "bias", None) is not None:
            nn.init.constant_(self.final_layer.linear.bias, 0)

    def unpatchify(self, x: torch.Tensor):
        C = self.out_channels
        P = self.x_embedder.patch_size[0]
        h = int(x.shape[1] ** 0.5)
        w = h
        assert h * w == x.shape[1]

        x = x.reshape(
            shape=(
                x.shape[0],
                h,
                w,
                P,
                P,
                C,
            )
        )
        x = torch.einsum("lhwpqc->lchpwq", x)
        imgs = x.reshape(shape=(x.shape[0], C, h * P, w * P))
        return imgs

    def forward(self, x, t, y):
        x = self.x_embedder(x) + self.pos_embed
        t = self.t_embedder(t, freq_scale=self.timestep_freq_scale)
        y = self.y_embedder(y, self.training)
        c = t + y
        for block in self.blocks:
            x = block(x, c=c)
        x = self.final_layer(x, c=c)
        x = self.unpatchify(x)
        return x


def get_2d_sincos_pos_embed(embed_dim, grid_size):
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)
    grid = np.stack(grid, axis=0)

    grid = grid.reshape([2, 1, grid_size, grid_size])
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    return pos_embed


def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])
    emb = np.concatenate([emb_h, emb_w], axis=1)
    return emb


def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    assert embed_dim % 2 == 0
    half = embed_dim // 2

    omega = np.arange(half, dtype=np.float64)
    omega /= half
    omega = 1.0 / (10000**omega)

    pos = pos.reshape(-1)
    out = np.einsum("m,d->md", pos, omega)

    emb_sin = np.sin(out)
    emb_cos = np.cos(out)

    emb = np.concatenate([emb_sin, emb_cos], axis=1)
    return emb


In [12]:
my_nano_dit = NanoDiT()

In [13]:
my_nano_dit(
    torch.randn((1, 3, 32, 32)),
    torch.tensor([1]),
    torch.tensor([9]),
).shape

torch.Size([1, 3, 32, 32])